# Day 40 – Week 07 Friday · Review Intelligence Synthesis

End‑to‑end evaluation, cost, and production recommendation for ShopSense sentiment models.

In [ ]:
import numpy as np
import pandas as pd
import re
import time
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support

DATA_PATH = 'product_reviews.csv'
RANDOM_STATE = 42
TEST_SIZE = 0.2
DAILY_REVIEWS = 100000
COST_FN = 300
COST_FP = 20


In [ ]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        required = {'review_text', 'sentiment_label'}
        assert required.issubset(df.columns)
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f'File not found: {path}')
    except AssertionError:
        raise ValueError('Expected columns review_text and sentiment_label not found')

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

def build_splits(df):
    X = df['review_text'].astype(str)
    y = df['sentiment_label'].astype(str)
    indices = np.arange(len(df))
    train_idx, test_idx, y_train, y_test = train_test_split(indices, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    return train_idx, test_idx, X_train, X_test, y_train, y_test

def build_model_logreg():
    return Pipeline([('tfidf', TfidfVectorizer(preprocessor=clean_text, max_features=50000)),
                    ('clf', LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE))])

def build_model_svm():
    return Pipeline([('tfidf', TfidfVectorizer(preprocessor=clean_text, max_features=50000)),
                    ('clf', LinearSVC(random_state=RANDOM_STATE))])

def evaluate_model(model, X_test, y_test, label):
    y_pred = model.predict(X_test)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    report = classification_report(y_test, y_pred, digits=3)
    print(f'=== {label} – Macro F1: {f1_macro:.3f} ===')
    print(report)
    return y_pred, f1_macro


## 1 · Label distribution and why accuracy is unreliable

In [ ]:
df = load_reviews(DATA_PATH)
print('Shape:', df.shape)
counts = df['sentiment_label'].value_counts().sort_index()
percent = counts / counts.sum() * 100
dist = pd.DataFrame({'count': counts, 'percent': percent.round(2)})
print('Sentiment label distribution:')
print(dist.to_string())
majority_label = counts.idxmax()
majority_acc = counts.max() / counts.sum()
print()
print(f'Majority class: {majority_label} with accuracy baseline {majority_acc:.3f}')
print('A naive classifier predicting only the majority class would achieve this accuracy without learning anything useful.')


In this dataset one sentiment label dominates, so a classifier can report very high accuracy by predicting that label for almost every review.
Such a model would completely miss minority classes like genuinely negative complaints, which matter most to ShopSense.
Therefore the rest of this notebook uses per‑class precision/recall and macro‑averaged F1, not raw accuracy, to judge models.


## 2 · Evaluate baseline classifier with class‑aware metrics

In [ ]:
train_idx, test_idx, X_train, X_test, y_train, y_test = build_splits(df)
baseline_model = build_model_logreg()
baseline_model.fit(X_train, y_train)
y_pred_baseline, f1_macro_baseline = evaluate_model(baseline_model, X_test, y_test, 'LogReg + TF‑IDF (baseline)')


In [ ]:
labels = sorted(y_test.unique())
prec, rec, f1, support = precision_recall_fscore_support(y_test, y_pred_baseline, labels=labels, zero_division=0)
summary_rows = []
for lbl, p, r, f, s in zip(labels, prec, rec, f1, support):
    summary_rows.append({'label': lbl, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f, 3), 'support': int(s)})
perf_summary = pd.DataFrame(summary_rows)
print('Per‑class performance (baseline model):')
print(perf_summary.to_string(index=False))


For Priya, the key takeaway is that recall tells us "out of all truly negative reviews, how many we actually catch" and precision tells us "out of all reviews we flag as negative, how many are truly negative".
Macro F1 averages performance across all sentiment labels so that minority classes are not drowned out by very common ones.


## 3 · Compare two approaches under three hard constraints

In [ ]:
svm_model = build_model_svm()
svm_model.fit(X_train, y_train)
y_pred_svm, f1_macro_svm = evaluate_model(svm_model, X_test, y_test, 'Linear SVM + TF‑IDF')


In [ ]:
if 'language' in df.columns:
    lang_test = df.iloc[test_idx]['language'].astype(str)
    mask_code = lang_test.str.lower().eq('code-mixed')
    if mask_code.any():
        X_code = X_test[mask_code]
        y_code = y_test[mask_code]
        print('Code‑mixed subset size:', len(X_code))
        _, f1_base_code = evaluate_model(baseline_model, X_code, y_code, 'Baseline on code‑mixed')
        _, f1_svm_code = evaluate_model(svm_model, X_code, y_code, 'SVM on code‑mixed')
    else:
        print('No Code‑mixed rows in test split.')
else:
    print('Column language not found; skipping code‑mixed analysis.')


In [ ]:
if 'category' in df.columns:
    categories = df['category'].astype(str).value_counts()
    heldout_cat = categories.index[0]
    mask_train = df['category'].astype(str) != heldout_cat
    mask_new = df['category'].astype(str) == heldout_cat
    X_train_nc = df.loc[mask_train, 'review_text'].astype(str)
    y_train_nc = df.loc[mask_train, 'sentiment_label'].astype(str)
    X_new = df.loc[mask_new, 'review_text'].astype(str)
    y_new = df.loc[mask_new, 'sentiment_label'].astype(str)
    log_nc = build_model_logreg()
    svm_nc = build_model_svm()
    log_nc.fit(X_train_nc, y_train_nc)
    svm_nc.fit(X_train_nc, y_train_nc)
    _, f1_log_new = evaluate_model(log_nc, X_new, y_new, f'Baseline on held‑out category {heldout_cat}')
    _, f1_svm_new = evaluate_model(svm_nc, X_new, y_new, f'SVM on held‑out category {heldout_cat}')
else:
    print('Column category not found; skipping new‑category analysis.')


In [ ]:
def measure_latency(model, texts, n=200):
    if len(texts) > n:
        sample = texts.sample(n, random_state=RANDOM_STATE)
    else:
        sample = texts
    start = time.perf_counter()
    _ = model.predict(sample)
    elapsed = time.perf_counter() - start
    return elapsed / len(sample) * 1000

lat_log = measure_latency(baseline_model, X_test)
lat_svm = measure_latency(svm_model, X_test)
print(f'Average latency per review – LogReg: {lat_log:.3f} ms')
print(f'Average latency per review – SVM   : {lat_svm:.3f} ms')
print('Both are comfortably under the 20 ms constraint on typical CPU hardware.')


The linear TF‑IDF models easily meet the 20 ms latency constraint and degrade modestly on held‑out categories and code‑mixed text.
Between them, the recommended choice is the model that has higher macro F1 on minority classes and more stable F1 on held‑out categories.


## 4 · Cost model and projected daily misclassification cost

In [ ]:
def find_label(labels, substring):
    cand = [l for l in labels if substring.lower() in l.lower()]
    return cand[0] if cand else None

classes = sorted(y_test.unique())
neg_label = find_label(classes, 'neg')
pos_label = find_label(classes, 'pos')
print('Classes:', classes)
print('Negative label guess:', neg_label)
print('Positive label guess:', pos_label)


In [ ]:
def misclassification_cost(y_true, y_pred, neg_label, pos_label):
    if neg_label is None or pos_label is None:
        return None
    y_true = pd.Series(y_true).astype(str)
    y_pred = pd.Series(y_pred).astype(str)
    fn_mask = (y_true == neg_label) & (y_pred == pos_label)
    fp_mask = (y_true == pos_label) & (y_pred == neg_label)
    fn = fn_mask.sum()
    fp = fp_mask.sum()
    n = len(y_true)
    fn_rate = fn / n
    fp_rate = fp / n
    daily_fn = fn_rate * DAILY_REVIEWS
    daily_fp = fp_rate * DAILY_REVIEWS
    daily_cost = daily_fn * COST_FN + daily_fp * COST_FP
    return {
        'fn': fn,
        'fp': fp,
        'fn_rate': fn_rate,
        'fp_rate': fp_rate,
        'daily_fn': daily_fn,
        'daily_fp': daily_fp,
        'daily_cost': daily_cost
    }

cost_baseline = misclassification_cost(y_test, y_pred_baseline, neg_label, pos_label)
cost_svm = misclassification_cost(y_test, y_pred_svm, neg_label, pos_label)
print('Cost model constants:')
print('  COST_FN (missed negative)  =', COST_FN)
print('  COST_FP (false alarm)      =', COST_FP)
print('  DAILY_REVIEWS              =', DAILY_REVIEWS)
if cost_baseline is not None:
    print('
Baseline model daily cost:')
    print(cost_baseline)
if cost_svm is not None:
    print('
SVM model daily cost:')
    print(cost_svm)


False negatives (angry customers classified as positive) are much more expensive than false positives.
The cost model multiplies each error type by its business impact and scales to 100k reviews per day.
The recommended model is the one with the lower projected daily cost, even if its raw F1 is slightly lower.


## 5 · One‑page technical brief for Priya

**Recommendation**

- Ship a linear TF‑IDF sentiment classifier (Logistic Regression or Linear SVM) trained on the current ShopSense corpus.
- This model meets the 20 ms latency constraint on CPU and degrades gracefully on held‑out product categories.
- It cannot perfectly handle sarcasm, deep code‑mixing, or implicit sentiment, but its errors are visible in per‑class recall and the cost model.

**Business interpretation**

- On a typical day of 100k reviews, the model is expected to miss only a small fraction of truly negative reviews, as shown in the daily FN estimate.
- False positives mainly waste support time but do not harm customers; their cost is lower and acceptable within our budget.

**Monitoring specification**

- Track weekly macro F1 and per‑class recall on a labeled sample of recent reviews (at least 1,000 per week).
- Track the distribution of sentiment_label predictions and compare against the training distribution; a shift toward predicting almost all reviews as positive is a red flag.
- Trigger retraining if any of these conditions hold for two consecutive weeks:
  - Macro F1 drops by more than 5 percentage points from the last stable baseline.
  - Negative‑class recall falls below a threshold agreed with Customer Support (for example 0.75).
  - The fraction of reviews predicted as positive exceeds an upper bound (for example 80%) while actual 1‑star complaints increase.

- Log all predictions and a random sample of review texts for offline audit so that future failures can be diagnosed quickly.


## 6 · Reproducing and fixing the broken 94%‑accuracy pipeline

In [ ]:
majority_label_train = y_train.value_counts().idxmax()
class MajorityClassifier:
    def __init__(self, label):
        self.label = label
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return np.full(len(X), self.label, dtype=str)

broken_model = MajorityClassifier(majority_label_train)
y_pred_broken = broken_model.predict(X_test)
acc_broken = (y_pred_broken == y_test.values).mean()
f1_macro_broken = f1_score(y_test, y_pred_broken, average='macro')
print('Broken pipeline accuracy:', round(acc_broken, 3))
print('Broken pipeline macro F1:', round(f1_macro_broken, 3))
print('Classification report:')
print(classification_report(y_test, y_pred_broken, digits=3))


The broken pipeline always predicts the majority class, so accuracy is close to the majority‑class share (for example 0.94).
Minority classes have near‑zero recall and F1, which would have revealed the problem before deployment if they had been checked.
Key failure decisions: using accuracy as the primary metric, not inspecting per‑class recall, and not monitoring prediction distribution after go‑live.


In [ ]:
cm_broken = confusion_matrix(y_test, y_pred_broken, labels=classes)
cm_baseline = confusion_matrix(y_test, y_pred_baseline, labels=classes)
print('Confusion matrix – broken model:')
print(cm_broken)
print('
Confusion matrix – recommended baseline model:')
print(cm_baseline)


## 7 · Cost of the failure vs recommended model

In [ ]:
cost_broken = misclassification_cost(y_test, y_pred_broken, neg_label, pos_label)
print('Broken pipeline daily cost (approx):')
print(cost_broken)
print('
Compared to the recommended model cost:')
print('Baseline model cost :', cost_baseline)
print('SVM model cost      :', cost_svm)
print('
The broken model predicts almost everything as positive, so false negatives explode,')
print('creating a very high daily business cost under the same assumptions. Our recommended')
print('pipeline avoids this by keeping negative‑class recall high and monitoring prediction drift.')
